# Linux Process Management

## Expanded Educational Notebook

This notebook provides a complete beginner-to-intermediate introduction to Linux process management.

It covers:

- programs, processes, tasks, and threads,
- PID and PPID,
- process ownership,
- process priorities,
- boot flow and PID 1,
- `ps` and its major option styles,
- process states,
- `pstree`,
- foreground and background jobs,
- signals and `kill`,
- `nice` and `renice`,
- zombies and orphan processes,
- `/proc`,
- `top` and `htop`,
- daemons and services,
- practical labs,
- review questions and answers.

## Learning objectives

After completing this notebook, you should be able to:

1. explain what a process is,
2. distinguish a program from a process,
3. explain PID and PPID,
4. identify the process owner,
5. interpret process priority and nice values,
6. use `ps` in several common forms,
7. interpret process states such as `R`, `S`, `T`, and `Z`,
8. display the process tree using `pstree`,
9. manage foreground and background jobs,
10. send signals to processes,
11. change process priority,
12. inspect processes through `/proc`,
13. explain zombies and orphan processes,
14. understand PID 1 and system services.

# Part I — Basic Concepts

## 1. What is a process?

A **process** is a running instance of a program.

A process includes:

- executable code,
- current CPU state,
- memory,
- open files,
- environment variables,
- process credentials,
- scheduling information,
- signal state.

Linux may also use the word **task** internally for schedulable execution units.

## 2. Program versus process

A **program** is a file stored on disk.

Examples:

```text
/bin/ls
/usr/bin/python3
/home/ali/script.sh
```

A **process** is created when a program is executed.

Example:

```bash
ls
```

The file `/bin/ls` is the program.  
The running instance is the process.

## 3. One program can create many processes

You may run the same program several times.

Example:

```bash
sleep 100 &
sleep 100 &
sleep 100 &
```

All three processes execute the same program but have different:

- PIDs,
- memory state,
- start times,
- parent relationships.

## 4. Linux is multitasking

Linux can manage many processes at the same time.

On a single CPU core, the kernel rapidly switches between runnable tasks.

On multicore systems, several processes may truly execute simultaneously.

This scheduling creates the experience of concurrent execution.

## 5. Process resources

A process may have:

- virtual memory,
- file descriptors,
- working directory,
- root directory,
- user and group IDs,
- signal handlers,
- CPU time,
- priority,
- parent process,
- child processes.

# Part II — PID and PPID

## 6. What is PID?

PID means:

> Process ID

The kernel assigns a PID to every running process.

Example:

```bash
ps
```

Possible output:

```text
PID TTY          TIME CMD
421 pts/0    00:00:00 bash
615 pts/0    00:00:00 ps
```

## 7. Are PIDs permanently unique?

No.

A PID is unique only while the process exists.

After a process exits, the kernel may reuse the number for a future process.

## 8. Current shell PID

In many shells:

```bash
echo $$
```

prints the PID of the current shell.

In [ ]:
echo "Current shell PID: $$"
ps -p $$ -o pid,ppid,user,stat,cmd

## 9. What is PPID?

PPID means:

> Parent Process ID

Most processes are created by another process.

The PPID identifies that parent.

## 10. Parent-child relationship

Example:

```text
terminal
  └── bash
       └── sleep
```

The shell starts `sleep`.

Therefore:

- shell PID = parent,
- `sleep` PPID = shell PID.

In [ ]:
sleep 60 &
child=$!
echo "Child PID: $child"
ps -p "$child" -o pid,ppid,user,stat,cmd
kill "$child"
wait "$child" 2>/dev/null || true

## 11. Process trees

Parent-child relationships form a process tree.

At the root of the userspace process tree is PID 1.

# Part III — Process Ownership

## 12. Every process has a user

A process runs with user credentials.

Important identities include:

- real user ID,
- effective user ID,
- saved user ID.

For basic administration, the effective user ID is especially important because it is used for many permission checks.

## 13. Process owner versus file owner

The user who owns a process does not need to own the program file.

Example:

```text
/usr/bin/ls
```

is normally owned by root.

But when user `ali` runs it, the process usually runs as `ali`.

## 14. Concrete example

Suppose:

```text
-rwxr-xr-x root root /usr/bin/cat
```

User `ali` runs:

```bash
cat notes.txt
```

The `cat` executable is owned by root, but the process normally runs as `ali`.

## 15. Why process ownership matters

Process ownership determines:

- which files it can access,
- which signals it may send,
- what system resources it may use,
- whether it may change priority,
- what privileged operations are allowed.

# Part IV — Process Creation

## 16. `fork()` and `exec()`

A simplified Unix process-creation model is:

1. the parent calls `fork()`,
2. a child process is created,
3. the child often calls `exec()`,
4. `exec()` replaces the child's program image.

Conceptually:

```text
bash
  |
 fork
  |
 child bash
  |
 exec ls
  |
 ls process
```

## 17. `fork()` does not simply “run a program”

`fork()` creates a new process based on the caller.

`exec()` loads a new program into that process.

This separation is a central Unix design idea.

## 18. Shell example

When you type:

```bash
ls
```

the shell generally:

1. creates a child process,
2. the child executes `ls`,
3. the parent shell waits,
4. `ls` exits,
5. the shell shows the prompt again.

# Part V — Boot Flow and PID 1

## 19. Correct boot sequence

A simplified sequence is:

```text
Firmware (BIOS/UEFI)
        ↓
Boot loader
        ↓
Linux kernel
        ↓
PID 1
        ↓
services and user sessions
```

## 20. Is the boot loader a process?

No, not in the normal Linux-process sense.

The boot loader runs before the Linux kernel starts process management.

Examples include:

- GRUB,
- systemd-boot,
- U-Boot.

## 21. Is the kernel a process?

The kernel is not one normal userspace process.

It manages processes and may expose kernel threads, but the kernel itself is the operating-system core.

## 22. PID 1

The first userspace process has PID 1.

Historically this was commonly called `init`.

On many modern systems, PID 1 is:

```text
systemd
```

In [ ]:
ps -p 1 -o pid,ppid,user,stat,comm,args

## 23. Responsibilities of PID 1

PID 1 commonly:

- starts system services,
- manages service dependencies,
- adopts orphaned processes,
- reaps some terminated child processes,
- coordinates shutdown and reboot.

# Part VI — `ps`

## 24. What does `ps` do?

`ps` displays a snapshot of process information.

It is not a continuously updating monitor.

For live updates, use tools such as:

- `top`,
- `htop`.

## 25. Plain `ps`

```bash
ps
```

Usually shows processes:

- owned by the current user,
- attached to the current terminal,
- including the shell and `ps` itself.

## 26. `ps -e`

```bash
ps -e
```

Shows all processes.

## 27. `ps -f`

```bash
ps -f
```

Shows a fuller format, often including:

- UID,
- PID,
- PPID,
- start time,
- command.

## 28. `ps -ef`

```bash
ps -ef
```

Shows all processes in full format.

## 29. `ps aux`

```bash
ps aux
```

This is BSD-style syntax.

It commonly shows:

- user,
- PID,
- CPU percentage,
- memory percentage,
- virtual memory,
- resident memory,
- terminal,
- state,
- start time,
- CPU time,
- command.

## 30. Why are there different `ps` styles?

`ps` supports several historical option styles:

- Unix style: `-e`, `-f`
- BSD style: `aux`
- GNU long style: `--sort`, `--forest`

These styles can sometimes be combined, but output behavior may differ.

## 31. Select a process by PID

```bash
ps -p 1234
```

Custom fields:

```bash
ps -p 1234 -o pid,ppid,user,stat,ni,pri,cmd
```

## 32. Select processes by user

```bash
ps -u alice
```

or:

```bash
ps -U alice
```

Exact semantics differ between effective and real user selection.

## 33. Custom output

```bash
ps -eo pid,ppid,user,ni,pri,stat,etime,cmd
```

This is one of the most useful process-inspection forms.

## 34. Sort output

Examples:

```bash
ps -eo pid,user,%cpu,%mem,cmd --sort=-%cpu
ps -eo pid,user,%cpu,%mem,cmd --sort=-%mem
```

A leading minus means descending order.

## 35. Show a process forest

```bash
ps -ef --forest
```

or:

```bash
ps axjf
```

This helps visualize parent-child relationships.

## 36. Common `ps` columns

| Column | Meaning |
|---|---|
| PID | Process ID |
| PPID | Parent PID |
| USER/UID | Process owner |
| STAT | Process state |
| NI | Nice value |
| PRI | Scheduler priority |
| %CPU | CPU usage estimate |
| %MEM | Memory usage percentage |
| VSZ | Virtual memory size |
| RSS | Resident memory |
| TTY | Controlling terminal |
| TIME | CPU time consumed |
| CMD/COMMAND | Command |

# Part VII — Process States

## 37. `R` state

`R` means:

> Running or runnable

The process is either:

- currently executing on a CPU,
- or ready to execute.

## 38. `S` state

`S` means:

> Interruptible sleep

The process is waiting for an event, such as:

- user input,
- network data,
- timer expiration,
- child completion.

This is a normal state.

## 39. `T` state

`T` means:

> Stopped or traced

A process may enter this state after:

- `Ctrl+Z`,
- `SIGSTOP`,
- debugger control.

## 40. `Z` state

`Z` means:

> Zombie

A zombie process has finished execution, but its parent has not yet collected its exit status.

## 41. Zombie processes do not keep normal execution resources

A zombie usually retains only a small process-table entry containing information such as:

- PID,
- exit status,
- accounting data.

It does not continue running.

## 42. `D` state

`D` means:

> Uninterruptible sleep

This often indicates waiting for kernel-level I/O.

Processes in `D` state may not respond immediately to signals, including `SIGKILL`.

## 43. `I` state

`I` commonly means:

> Idle kernel thread

This state appears on many modern Linux systems.

## 44. Extra state characters

`STAT` may contain more than one character.

Examples:

- `S+`
- `Ssl`
- `R<`

Additional flags may indicate:

- foreground process group,
- multithreading,
- high priority,
- session leader.

# Part VIII — `pstree`

## 45. What is `pstree`?

`pstree` displays processes in a tree format.

It makes PPID relationships easier to understand.

## 46. Basic usage

```bash
pstree
```

## 47. Show PIDs

```bash
pstree -p
```

## 48. Show users

```bash
pstree -u
```

## 49. Show one branch

```bash
pstree -p $$
```

This displays the current shell branch.

## 50. If `pstree` is missing

On Debian/Ubuntu:

```bash
sudo apt install psmisc
```

On Fedora:

```bash
sudo dnf install psmisc
```

# Part IX — Foreground and Background Jobs

## 51. Foreground process

A foreground process controls the terminal.

Example:

```bash
sleep 100
```

The shell waits until it finishes.

## 52. Background process

Add `&`:

```bash
sleep 100 &
```

The shell returns immediately.

## 53. `$!`

After starting a background process:

```bash
sleep 100 &
echo $!
```

`$!` contains the PID of the most recent background job.

## 54. `jobs`

```bash
jobs
```

Shows jobs managed by the current shell.

## 55. `Ctrl+Z`

`Ctrl+Z` normally sends `SIGTSTP` to the foreground job.

The job becomes stopped.

## 56. `bg`

```bash
bg %1
```

Continues a stopped job in the background.

## 57. `fg`

```bash
fg %1
```

Brings a job to the foreground.

## 58. Job ID versus PID

A shell job ID looks like:

```text
%1
```

A process PID looks like:

```text
4321
```

They are not the same.

# Part X — Signals

## 59. What is a signal?

A signal is an asynchronous notification sent to a process.

Signals can request actions such as:

- terminate,
- stop,
- continue,
- reload configuration,
- handle an event.

## 60. List signals

```bash
kill -l
```

## 61. `SIGTERM`

Signal 15:

```bash
kill PID
```

or:

```bash
kill -TERM PID
```

This politely requests termination.

## 62. `SIGKILL`

Signal 9:

```bash
kill -KILL PID
```

or:

```bash
kill -9 PID
```

The process cannot catch or ignore it.

Use only when graceful termination fails.

## 63. `SIGSTOP`

```bash
kill -STOP PID
```

Stops a process unconditionally.

## 64. `SIGCONT`

```bash
kill -CONT PID
```

Continues a stopped process.

## 65. `SIGHUP`

Historically means terminal hangup.

Many daemons use it as a request to reload configuration.

## 66. `SIGINT`

Usually sent by:

```text
Ctrl+C
```

It requests interruption.

## 67. `kill` does not always mean “kill”

The command name is misleading.

`kill` sends signals.

Example:

```bash
kill -STOP PID
```

does not terminate the process.

## 68. `pkill`

```bash
pkill firefox
```

Matches processes by name or other attributes.

Use carefully because multiple processes may match.

## 69. `killall`

```bash
killall firefox
```

On Linux, this usually sends a signal to processes matching a name.

Behavior differs on some Unix systems, so be careful.

# Part XI — Priority and Nice Values

## 70. What is process priority?

The scheduler decides which runnable process gets CPU time.

Priority information helps influence this decision.

## 71. Nice value

The nice value usually ranges from:

```text
-20 to 19
```

- `-20` = less nice to others, higher priority,
- `19` = more nice to others, lower priority.

## 72. Default nice value

Most normal processes start with nice value:

```text
0
```

## 73. Start with a chosen nice value

```bash
nice -n 10 command
```

Example:

```bash
nice -n 10 gzip largefile
```

## 74. Change an existing process priority

```bash
renice 10 -p PID
```

## 75. Permission rules

Ordinary users can usually:

- increase the nice value,
- reduce their own process priority.

Raising priority by lowering the nice value generally requires privilege.

## 76. `PRI` versus `NI`

`NI` is the user-visible nice value.

`PRI` is scheduler priority information and may be displayed differently across tools.

They are related but not identical.

# Part XII — `top` and `htop`

## 77. `top`

`top` provides a continuously updating process display.

Run:

```bash
top
```

## 78. Important `top` fields

Common fields:

- load average,
- task counts,
- CPU usage,
- memory usage,
- swap usage,
- process table.

## 79. Useful `top` keys

Common interactive keys:

| Key | Action |
|---|---|
| `q` | quit |
| `P` | sort by CPU |
| `M` | sort by memory |
| `k` | send signal |
| `r` | renice |
| `1` | show per-CPU data |
| `H` | toggle threads |

## 80. `htop`

`htop` is an interactive alternative with:

- easier navigation,
- colored display,
- process tree mode,
- mouse support,
- simpler signal and priority controls.

It may need separate installation.

# Part XIII — `/proc`

## 81. What is `/proc`?

`/proc` is a virtual filesystem exposing kernel and process information.

Each process usually has a directory:

```text
/proc/PID/
```

## 82. Inspect the current shell

```bash
ls /proc/$$
```

## 83. `/proc/PID/status`

```bash
cat /proc/PID/status
```

Shows readable process information such as:

- name,
- state,
- PID,
- PPID,
- UID,
- GID,
- memory statistics,
- capabilities.

## 84. `/proc/PID/cmdline`

```bash
tr '\0' ' ' < /proc/PID/cmdline
```

Displays command-line arguments.

## 85. `/proc/PID/environ`

```bash
tr '\0' '\n' < /proc/PID/environ
```

Shows environment variables, subject to permissions.

## 86. `/proc/PID/fd`

```bash
ls -l /proc/PID/fd
```

Shows open file descriptors.

## 87. `/proc/PID/cwd`

```bash
readlink /proc/PID/cwd
```

Shows the process working directory.

## 88. `/proc/PID/exe`

```bash
readlink /proc/PID/exe
```

Shows the executable path.

# Part XIV — Zombies and Orphans

## 89. What is a zombie?

A child exits.

The kernel keeps its exit status until the parent calls a wait function.

During this period, the child is a zombie.

## 90. Can you kill a zombie?

No meaningful signal can make an already-dead process exit again.

The real issue is the parent not reaping it.

## 91. How zombies disappear

A zombie disappears when:

- the parent calls `wait()` or similar,
- or the parent exits and PID 1/subreaper adopts and reaps the child.

## 92. What is an orphan process?

An orphan is a still-running process whose parent exits.

It is adopted by PID 1 or another designated subreaper.

## 93. Zombie versus orphan

| Zombie | Orphan |
|---|---|
| already exited | still running |
| waiting to be reaped | parent has exited |
| state `Z` | may be `S`, `R`, etc. |

# Part XV — Daemons and Services

## 94. What is a daemon?

A daemon is a long-running background process that provides a service.

Examples:

- web server,
- SSH server,
- logging service,
- scheduler.

## 95. Daemon naming

Historically many daemon names end with `d`.

Examples:

```text
sshd
cron
systemd
```

Not every daemon follows this convention.

## 96. Service management with systemd

Common commands:

```bash
systemctl status ssh
systemctl start ssh
systemctl stop ssh
systemctl restart ssh
systemctl enable ssh
systemctl disable ssh
```

Service names vary by distribution.

## 97. Process versus service

A process is a running execution instance.

A service is a managed system function that may involve one or more processes.

# Part XVI — Threads

## 98. Process versus thread

A process has its own virtual address space.

Threads within one process share much of that address space but have separate execution contexts.

## 99. Show threads with `ps`

```bash
ps -eLf
```

or:

```bash
ps -T -p PID
```

## 100. Threads in `top`

Press:

```text
H
```

to toggle thread display.

# Part XVII — Practical Labs

## 101. Lab setup

In [ ]:
echo "Shell PID: $$"
echo "Shell PPID: $PPID"
ps -p $$ -o pid,ppid,user,ni,pri,stat,etime,cmd

## 102. Lab: create and inspect a child

In [ ]:
sleep 120 &
pid=$!

echo "Started PID=$pid"
ps -p "$pid" -o pid,ppid,user,ni,pri,stat,etime,cmd

## 103. Lab: inspect through `/proc`

In [ ]:
echo "--- status ---"
grep -E '^(Name|State|Pid|PPid|Uid|Gid):' /proc/$pid/status

echo
echo "--- executable ---"
readlink /proc/$pid/exe

echo
echo "--- cwd ---"
readlink /proc/$pid/cwd

## 104. Lab: stop and continue

In [ ]:
kill -STOP "$pid"
sleep 1
ps -p "$pid" -o pid,ppid,stat,cmd

kill -CONT "$pid"
sleep 1
ps -p "$pid" -o pid,ppid,stat,cmd

## 105. Lab: change nice value

In [ ]:
renice 10 -p "$pid"
ps -p "$pid" -o pid,ni,pri,stat,cmd

## 106. Lab: terminate gracefully

In [ ]:
kill -TERM "$pid"
wait "$pid" 2>/dev/null || true
echo "Process finished."

## 107. Lab: compare `ps` styles

In [ ]:
echo "--- ps ---"
ps

echo
echo "--- ps -ef (first rows) ---"
ps -ef | head

echo
echo "--- ps aux (first rows) ---"
ps aux | head

echo
echo "--- custom output ---"
ps -eo pid,ppid,user,ni,pri,stat,%cpu,%mem,etime,cmd --sort=pid | head -20

## 108. Lab: process tree

In [ ]:
pstree -p $$ 2>/dev/null || echo "pstree is not installed"

## 109. Lab: background jobs

In [ ]:
sleep 30 &
jobpid=$!

jobs
echo "Background PID=$jobpid"

kill "$jobpid"
wait "$jobpid" 2>/dev/null || true

# Part XVIII — Common Mistakes

## 110. Mistake: saying boot loader, kernel, and init are the first three processes

This is incorrect.

- the boot loader is not a Linux process,
- the kernel is not a normal userspace process,
- PID 1 is the first userspace process.

## 111. Mistake: assuming process owner equals executable owner

The executable may be owned by root while the process runs as an ordinary user.

## 112. Mistake: thinking `kill` always terminates

`kill` sends a signal.

The selected signal decides the requested action.

## 113. Mistake: using `kill -9` first

Prefer graceful termination:

```bash
kill PID
```

Use `kill -9` only when necessary.

## 114. Mistake: assuming a zombie is still executing

A zombie has already exited.

It is waiting for its parent to collect its status.

## 115. Mistake: confusing job IDs and PIDs

`%1` is a shell job ID.

`1234` is a PID.

# Part XIX — Review Questions

## 116. Questions

1. What is a process?
2. What is the difference between a program and a process?
3. What does PID mean?
4. Can PID numbers be reused?
5. What does PPID mean?
6. What is the relationship between a shell and a command it starts?
7. Does a process owner have to own the executable file?
8. What is PID 1?
9. Is the boot loader a Linux process?
10. What does plain `ps` usually show?
11. What does `ps -e` show?
12. What does `ps -ef` show?
13. What does `ps aux` show?
14. What is the meaning of `R`?
15. What is the meaning of `S`?
16. What is the meaning of `T`?
17. What is the meaning of `Z`?
18. What does `D` mean?
19. What does `pstree` do?
20. What does `pstree -p` add?
21. What is a foreground process?
22. What does `&` do?
23. What does `$!` contain?
24. What does `Ctrl+Z` normally do?
25. What does `bg` do?
26. What does `fg` do?
27. What is a signal?
28. What signal is sent by plain `kill PID`?
29. Why is `SIGKILL` special?
30. What does `SIGSTOP` do?
31. What does `SIGCONT` do?
32. What is a nice value?
33. Which has higher priority: nice `-10` or nice `10`?
34. What does `renice` do?
35. What is `/proc/PID/status`?
36. What is a zombie?
37. What is an orphan?
38. Can a zombie be killed normally?
39. What is a daemon?
40. What is the difference between a process and a service?

# Part XX — Answers

## 117. Answers

1. A running instance of a program.
2. A program is stored code; a process is active execution.
3. Process ID.
4. Yes, after the old process exits.
5. Parent Process ID.
6. The command process is usually a child of the shell.
7. No.
8. The first userspace process, usually systemd on modern systems.
9. No.
10. Processes associated with the current terminal/user context.
11. All processes.
12. All processes in full format.
13. A BSD-style detailed listing.
14. Running or runnable.
15. Interruptible sleep.
16. Stopped or traced.
17. Zombie.
18. Uninterruptible sleep.
19. Displays process parent-child relationships.
20. PIDs.
21. A process controlling the terminal.
22. Starts a command in the background.
23. PID of the most recent background job.
24. Stops the foreground job.
25. Continues a stopped job in the background.
26. Brings a job to the foreground.
27. An asynchronous process notification.
28. SIGTERM.
29. It cannot be caught or ignored.
30. Stops the process.
31. Continues a stopped process.
32. A user-visible scheduling preference.
33. Nice `-10`.
34. Changes nice value of an existing process.
35. A readable process-information file.
36. An exited process waiting to be reaped.
37. A running process whose parent exited.
38. No; it is already dead.
39. A long-running background service process.
40. A process is an execution instance; a service is a managed system function.

# Part XXI — Cheat Sheet

## 118. Process inspection

```bash
ps
ps -e
ps -ef
ps aux
ps -p PID
ps -u USER
ps -eo pid,ppid,user,ni,pri,stat,%cpu,%mem,etime,cmd
ps -ef --forest

pstree
pstree -p
pstree -u
```

## 119. Jobs and signals

```bash
command &
jobs
fg %1
bg %1

kill PID
kill -TERM PID
kill -KILL PID
kill -STOP PID
kill -CONT PID
kill -l
```

## 120. Priority

```bash
nice -n 10 command
renice 10 -p PID
```

## 121. `/proc`

```bash
cat /proc/PID/status
tr '\0' ' ' < /proc/PID/cmdline
ls -l /proc/PID/fd
readlink /proc/PID/cwd
readlink /proc/PID/exe
```

## 122. Most important ideas

```text
Program = executable file
Process = running program
PID = process identifier
PPID = parent process identifier
PID 1 = first userspace process
R = running/runnable
S = sleeping
T = stopped
Z = zombie
```